# Figure S27 — TAS diagrams for dacitic melt inclusions


## 1. Imports and shared 00_Setup

Arial, 8-point ordinary text, 9-point bold uppercase panel labels, full plot boxes, light gridlines, 300 dpi PNG export, and editable PDF/SVG text. This notebook is linear and contains no custom functions.


In [1]:
from pathlib import Path
import hashlib
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from PIL import Image
import tasplot

SINGLE_COLUMN_WIDTH_IN = 3.55
DOUBLE_COLUMN_WIDTH_IN = 7.25
MAXIMUM_FIGURE_HEIGHT_IN = 7.8
FIGURE_WIDTH_IN = DOUBLE_COLUMN_WIDTH_IN
FIGURE_HEIGHT_IN = 5.0
BASE_TEXT_SIZE_PT = 8.0
PANEL_LABEL_SIZE_PT = 9.0
MINIMUM_LINE_WIDTH_PT = 0.28
EXPORT_DPI = 300
PUBLICATION_FONT = "Arial"

installed_font_names = {entry.name for entry in font_manager.fontManager.ttflist}
if PUBLICATION_FONT not in installed_font_names:
    raise RuntimeError("Arial is required by 00_Setup.")
mpl.rcParams.update({
    "font.family":"sans-serif", "font.sans-serif":[PUBLICATION_FONT],
    "font.size":BASE_TEXT_SIZE_PT, "axes.labelsize":BASE_TEXT_SIZE_PT,
    "axes.titlesize":BASE_TEXT_SIZE_PT, "xtick.labelsize":BASE_TEXT_SIZE_PT,
    "ytick.labelsize":BASE_TEXT_SIZE_PT, "legend.fontsize":6.0,
    "legend.title_fontsize":BASE_TEXT_SIZE_PT, "figure.facecolor":"white",
    "axes.facecolor":"white", "axes.spines.top":True, "axes.spines.right":True,
    "axes.linewidth":0.5, "lines.linewidth":0.75,
    "xtick.direction":"in", "ytick.direction":"in",
    "xtick.major.width":0.5, "ytick.major.width":0.5,
    "savefig.dpi":EXPORT_DPI, "savefig.facecolor":"white",
    "savefig.transparent":False, "pdf.fonttype":42, "ps.fonttype":42,
    "svg.fonttype":"none",
})


## 2. Paths, S1–S3 workbooks, colors, and markers

This figure reads the two S3 compilation sheets and the S1 host/inclusion sheet. S2 is checked as part of the shared package but is not used for this figure. No legacy or outside data are read.


In [2]:
PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT = PROJECT_ROOT_OVERRIDE
if PROJECT_ROOT is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "SA_MI_Draft_FinalFigures_Work" / "DataSheets").is_dir():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project or set PROJECT_ROOT_OVERRIDE.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORK_ROOT = PROJECT_ROOT / "SA_MI_Draft_FinalFigures_Work"
FIGURE_DIRECTORY = WORK_ROOT / "Nov6_AllFigures" / "Figure_S27_Compilation_TAS"
OUTPUT_DIRECTORY = FIGURE_DIRECTORY / "Figure_S27_Compilation_TAS_Final"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

S1_PATH = WORK_ROOT / "DataSheets" / "Updated_Oct20" / "Dechert_S1_AllData.xlsx"
S2_PATH = WORK_ROOT / "DataSheets" / "Updated_Oct20" / "Dechert_S2_Standards.xlsx"
S3_PATH = WORK_ROOT / "DataSheets" / "Updated_Oct20" / "Dechert_S3_Compilations.xlsx"
for workbook_path in [S1_PATH, S2_PATH, S3_PATH]:
    if not workbook_path.is_file():
        raise FileNotFoundError(f"Missing supplemental workbook: {workbook_path}")
SOURCE_WORKBOOKS = {"S1":S1_PATH, "S2":S2_PATH, "S3":S3_PATH}
SOURCE_HASHES_BEFORE = {label:hashlib.sha256(path.read_bytes()).hexdigest() for label,path in SOURCE_WORKBOOKS.items()}

VOLCANO_COLORS = {
    "Altiplano-Puna Volcanic Complex":"#D4A017", "Dzarta-Kuduk Paleovolcano":"#8C4B2D",
    "Glacier Peak":"#ff1493", "Ichinsky(Ichinskii)":"#1F4E9E",
    "Iturup (Itirup)":"#7B2CBF", "Lassen":"#00796B", "Mazama":"#E85D04",
    "Mount St. Helens":"#5145CD", "Mt Hood":"#168AAD", "Pinatubo":"#C1121F",
    "Plat Pays":"#00A896", "Popocatepetl":"#7F1D1D", "Santorini":"#6A0572",
    "Suphan Dag":"#2E7D32", "Unzen":"#708238",
}
VOLCANO_MARKERS = {
    "Glacier Peak":"o", "Altiplano-Puna Volcanic Complex":"s",
    "Dzarta-Kuduk Paleovolcano":"^", "Ichinsky(Ichinskii)":"D",
    "Iturup (Itirup)":"v", "Lassen":"P", "Mazama":"X",
    "Mount St. Helens":"*", "Mt Hood":"<", "Pinatubo":">",
    "Plat Pays":"p", "Popocatepetl":"h", "Santorini":"H",
    "Suphan Dag":"8", "Unzen":"d",
}
POINT_SIZE = 12.0
SILICA_CUTOFF = 60.0
CO2_MAX_TRUSTED = 1000.0
GLOBAL_MIN_ELIGIBLE_N = 10
MISSING_TOKENS = {"":np.nan, " ":np.nan, "nd":np.nan, "ND":np.nan, "n.d.":np.nan, "N.D.":np.nan}


## 3. Read the source sheets and prepare the established TAS cohorts


In [3]:
global_raw = pd.read_excel(S3_PATH, sheet_name="GEOROC_GlobalDacites")
cascades_raw = pd.read_excel(S3_PATH, sheet_name="Wieser2023_MI_Compilation")
gp_raw = pd.read_excel(S1_PATH, sheet_name="MI_Hosts_Plag_Pxy")

global_all = pd.DataFrame(index=global_raw.index)
global_all["dataset"] = "Global dacites"
global_all["source"] = global_raw["CITATION"].fillna("Source not reported").astype(str)
global_location = global_raw["LOCATION"].astype("string").fillna("Location not reported")
global_all["volcano"] = global_location.str.rsplit("/", n=1).str[-1].str.strip().str.title()
global_all["volcano"] = global_all["volcano"].replace({"Mount Mazama":"Mazama", "Mt. Mazama":"Mazama", "Heisei Lavas":"Unzen"})
global_columns = {
    "h2o_wt":"H2O(WT%): altern. values or methods",
    "co2_ppm":"CO2(PPM)",
    "sio2_wt":"SIO2(WT%): altern. values or methods",
    "na2o_wt":"NA2O(WT%): altern. values or methods",
    "k2o_wt":"K2O(WT%): altern. values or methods",
}
for output_column, source_column in global_columns.items():
    global_all[output_column] = pd.to_numeric(global_raw[source_column].replace(MISSING_TOKENS), errors="coerce")
global_all["total_alkalis_wt"] = global_all["na2o_wt"] + global_all["k2o_wt"]
global_h2oco2_eligible = global_all["h2o_wt"].notna() & global_all["h2o_wt"].ge(0) & ~global_all["co2_ppm"].gt(CO2_MAX_TRUSTED)
global_h2oco2_counts = global_h2oco2_eligible.groupby(global_all["volcano"]).sum()
global_systems_retained = global_h2oco2_counts.loc[global_h2oco2_counts.ge(GLOBAL_MIN_ELIGIBLE_N)].index
global_frame = global_all.loc[global_all["volcano"].isin(global_systems_retained)].copy()

cascades_all = pd.DataFrame(index=cascades_raw.index)
cascades_all["dataset"] = "Cascades >60 wt% SiO2"
cascades_all["source"] = cascades_raw["reference"].fillna("Source not reported").astype(str)
cascades_all["volcano"] = cascades_raw["volcano"].astype("string").fillna("Location not reported").str.strip()
cascades_columns = {"sio2_wt":"sio2_raw_wt", "na2o_wt":"na2o_raw_wt", "k2o_wt":"k2o_raw_wt"}
for output_column, source_column in cascades_columns.items():
    cascades_all[output_column] = pd.to_numeric(cascades_raw[source_column].replace(MISSING_TOKENS), errors="coerce")
cascades_all["total_alkalis_wt"] = cascades_all["na2o_wt"] + cascades_all["k2o_wt"]
cascades_frame = cascades_all.loc[cascades_all["sio2_wt"].gt(SILICA_CUTOFF)].copy()

gp_discard_mask = gp_raw["Discard_SIMS"].astype("string").str.strip().str.casefold().eq("yes").fillna(False)
gp_matrix_mask = gp_raw["base_name"].astype("string").str.contains(r"mtx|matrix", case=False, na=False)
gp_work = gp_raw.loc[~gp_discard_mask & ~gp_matrix_mask].copy()
gp_all = pd.DataFrame(index=gp_work.index)
gp_all["dataset"] = "Glacier Peak"
gp_all["source"] = "This study"
gp_all["volcano"] = "Glacier Peak"
gp_all["sio2_wt"] = pd.to_numeric(gp_work["SiO2_SEMGlass_AnhydoNorm"].replace(MISSING_TOKENS), errors="coerce")
gp_all["na2o_wt"] = pd.to_numeric(gp_work["Na2O_SEMGlass_AnhydoNorm"].replace(MISSING_TOKENS), errors="coerce")
gp_all["k2o_wt"] = pd.to_numeric(gp_work["K2O_SEMGlass_AnhydoNorm"].replace(MISSING_TOKENS), errors="coerce")
gp_all["total_alkalis_wt"] = gp_all["na2o_wt"] + gp_all["k2o_wt"]
gp_frame = gp_all.loc[gp_all["sio2_wt"].gt(SILICA_CUTOFF)].copy()

global_valid = global_frame[["sio2_wt","na2o_wt","k2o_wt","total_alkalis_wt"]].notna().all(axis=1) & global_frame["total_alkalis_wt"].ge(0)
cascades_valid = cascades_frame[["sio2_wt","na2o_wt","k2o_wt","total_alkalis_wt"]].notna().all(axis=1) & cascades_frame["total_alkalis_wt"].ge(0)
gp_valid = gp_frame[["sio2_wt","na2o_wt","k2o_wt","total_alkalis_wt"]].notna().all(axis=1) & gp_frame["total_alkalis_wt"].ge(0)
global_tas = global_frame.loc[global_valid].copy()
cascades_tas = cascades_frame.loc[cascades_valid].copy()
gp_tas = gp_frame.loc[gp_valid].copy()
panel_volcanoes = set(global_tas["volcano"]) | set(cascades_tas["volcano"]) | {"Glacier Peak"}
if not panel_volcanoes.issubset(VOLCANO_COLORS) or not panel_volcanoes.issubset(VOLCANO_MARKERS):
    raise ValueError("A plotted volcanic system is missing a configured color or marker.")


## 4. Plot and export Figure S27


In [4]:
FIGURE_STEM = "Figure_S27_TAS_Global_Dacites_Cascades"
figure, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN), sharex=True, sharey=True)
figure.subplots_adjust(left=0.075, right=0.985, top=0.95, bottom=0.23, wspace=0.11)

for axis, literature_frame, panel_title in zip(
    axes,
    [global_tas, cascades_tas],
    ["A) Global dacite melt inclusions", r"B) Cascades melt inclusions (SiO$_2$ > 60 wt%)"],
):
    tasplot.add_LeMaitre_fields(axis, fontsize=6.0, color="0.66")
    for volcano, points in literature_frame.groupby("volcano"):
        axis.scatter(points["sio2_wt"], points["total_alkalis_wt"], marker=VOLCANO_MARKERS[volcano],
                      s=POINT_SIZE, facecolors=VOLCANO_COLORS[volcano], edgecolors="black",
                      linewidths=MINIMUM_LINE_WIDTH_PT, alpha=0.82, zorder=3)
    for volcano, points in gp_tas.groupby("volcano"):
        axis.scatter(points["sio2_wt"], points["total_alkalis_wt"], marker=VOLCANO_MARKERS[volcano],
                      s=POINT_SIZE, facecolors=VOLCANO_COLORS[volcano], edgecolors="black",
                      linewidths=MINIMUM_LINE_WIDTH_PT, alpha=0.90, zorder=5)
    axis.set_xlim(35, 82)
    axis.set_ylim(0, 16)
    axis.set_xlabel(r"SiO$_2$ (wt%)")
    axis.set_title(panel_title, loc="left", pad=4.0, fontsize=PANEL_LABEL_SIZE_PT, fontweight="bold")
    axis.grid(True, color="0.88", linewidth=0.40, zorder=-5)
    axis.set_axisbelow(True)
    axis.tick_params(direction="in", top=True, right=True)
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.5)
axes[0].set_ylabel(r"Na$_2$O + K$_2$O (wt%)")

combined_names = ["Glacier Peak"] + sorted(panel_volcanoes - {"Glacier Peak"}, key=str.casefold)
legend_handles = [
    Line2D([0],[0], linestyle="none", marker=VOLCANO_MARKERS[name], markersize=4.3,
           markerfacecolor=VOLCANO_COLORS[name], markeredgecolor="black",
           markeredgewidth=MINIMUM_LINE_WIDTH_PT, label=name)
    for name in combined_names
]
figure_legend = figure.legend(handles=legend_handles, title="Volcanic Systems",
                              loc="upper center", bbox_to_anchor=(0.50, 0.135), ncol=4,
                              fontsize=6.0, title_fontsize=6.0, frameon=True,
                              fancybox=False, framealpha=1.0, facecolor="white",
                              edgecolor="black", borderpad=0.35, handlelength=1.15,
                              handletextpad=0.35, columnspacing=0.85,
                              labelspacing=0.18, borderaxespad=0)

SAVED_PATHS = {}
for extension in ["pdf", "png", "svg"]:
    output_path = OUTPUT_DIRECTORY / f"{FIGURE_STEM}.{extension}"
    save_options = {"format":extension, "facecolor":"white", "transparent":False}
    if extension == "png":
        save_options["dpi"] = EXPORT_DPI
    figure.savefig(output_path, **save_options)
    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise IOError(f"Expected export was not created: {output_path}")
    SAVED_PATHS[extension] = output_path
plt.close(figure)


## 5. Validate filters, layout, and exports


In [5]:
assert len(global_raw) == 450
assert len(cascades_raw) == 756
assert len(gp_raw) == 212
assert int(gp_discard_mask.sum()) == 13
assert int(gp_matrix_mask.sum()) == 13
assert int((gp_discard_mask & gp_matrix_mask).sum()) == 0
assert len(gp_all) == 186
assert (cascades_tas["sio2_wt"] > SILICA_CUTOFF).all()
assert figure_legend.get_title().get_text() == "Volcanic Systems"
assert len(figure_legend.get_texts()) == len(combined_names)
assert all(axis.get_title(loc="left").startswith(label) for axis,label in zip(axes,["A)","B)"]))
assert all(tuple(axis.get_xlim()) == (35.0,82.0) for axis in axes)
assert all(tuple(axis.get_ylim()) == (0.0,16.0) for axis in axes)
assert all(all(spine.get_visible() for spine in axis.spines.values()) for axis in axes)
with Image.open(SAVED_PATHS["png"]) as image:
    png_dimensions = image.size
assert png_dimensions == (2175,1500)
svg_text = SAVED_PATHS["svg"].read_text(errors="replace")
assert "<text" in svg_text and "<image" not in svg_text
SOURCE_HASHES_AFTER = {label:hashlib.sha256(path.read_bytes()).hexdigest() for label,path in SOURCE_WORKBOOKS.items()}
assert SOURCE_HASHES_AFTER == SOURCE_HASHES_BEFORE
print("Figure S27 complete.")
print("Valid TAS points:", {"global":len(global_tas), "cascades":len(cascades_tas), "Glacier Peak":len(gp_tas)})
print("Outputs:", {key:str(value) for key,value in SAVED_PATHS.items()})


Figure S27 complete.
Valid TAS points: {'global': 315, 'cascades': 322, 'Glacier Peak': 112}
Outputs: {'pdf': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_S27_Compilation_TAS/Figure_S27_Compilation_TAS_Final/Figure_S27_TAS_Global_Dacites_Cascades.pdf', 'png': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_S27_Compilation_TAS/Figure_S27_Compilation_TAS_Final/Figure_S27_TAS_Global_Dacites_Cascades.png', 'svg': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_S27_Compilation_TAS/Figure_S27_Compilation_TAS_Final/Figure_S27_TAS_Global_Dacites_Cascades.svg'}
